# Comprehensive Exploratory Data Analysis of Human Trafficking Reports to the Disque 100 Hotline

## Step 2: Data Quality Assessment

This notebook evaluates the quality of the administrative data before any substantive analysis of human rights violations. It preserves the source dataset, produces auditable diagnostics, and separates **true quality defects** from **structural or conditional missingness**.

### Scope

1. Schema and type conformance
2. Real and text-based missingness
3. Conditional missingness
4. Whitespace, Unicode, capitalization, and accent inconsistencies
5. Duplicate structures and dataset grain
6. Column-domain violations
7. Geographic consistency
8. Date and quantity validity
9. Cross-field business rules
10. Interactive Plotly quality charts
11. Quality-issue register and export


## 2.1 Import libraries and configure outputs

The notebook is standalone, but it follows the same loading conventions as Step 1. Change `DATA_PATH` only. The source is copied to `df_raw` and never overwritten.


In [ ]:
from pathlib import Path
import re
import unicodedata
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 150)
pd.set_option("display.max_colwidth", 140)

DATA_PATH = Path(r"../data/trafico-pessoas-com-outras-violacoes-2021-2025.csv")
SHEET_NAME = 0
CSV_ENCODING = "utf-8"
CSV_SEPARATOR = ";"
OUTPUT_DIR = Path("outputs/step_02_data_quality")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Pandas:", pd.__version__)


Pandas: 3.1.1


In [ ]:
from urllib.parse import urlparse
# Dados hospedados no GitHub
# DATA_PATH = "https://raw.githubusercontent.com/janaelsondias/eda-disque100/refs/heads/main/data/trafico-pessoas-com-outras-violacoes-2021-2025.csv"
DATA_PATH = Path("../data/trafico-pessoas-com-outras-violacoes-2021-2025.csv")
SHEET_NAME = 0
CSV_ENCODING = "utf-8"
CSV_SEPARATOR = ";"

def load_dataset(path, sheet_name=0, encoding="utf-8", separator=None):
    source = str(path)
    parsed = urlparse(source)
    is_url = parsed.scheme.lower() in {"http", "https"}
    ext = Path(parsed.path if is_url else source).suffix.lower()

    if ext == ".csv":
        options = {"sep": separator, "engine": "python" if separator is None else "c", "low_memory": False}
        if separator is not None:
            options["low_memory"] = False
        try:
            return pd.read_csv(source, encoding=encoding, **options)
        except UnicodeDecodeError:
            return pd.read_csv(source, encoding="latin-1", **options)

    if ext in {".xlsx", ".xls"}:
        return pd.read_excel(source, sheet_name=sheet_name)
    if ext == ".parquet":
        return pd.read_parquet(source)
    raise ValueError("Supported formats: CSV, XLSX/XLS, and Parquet")

is_url = urlparse(str(DATA_PATH)).scheme.lower() in {"http", "https"}
if is_url or Path(DATA_PATH).exists():
    df_raw = load_dataset(DATA_PATH, SHEET_NAME, CSV_ENCODING, CSV_SEPARATOR)
    df = df_raw.copy(deep=True)
    print(f"Loaded {len(df):,} rows and {df.shape[1]:,} columns")
else:
    df_raw = df = None
    print(f"Update DATA_PATH. File not found: {Path(DATA_PATH).resolve()}")


Loaded 8,401 rows and 62 columns


In [ ]:

import pandas as pd

# DataFrame analítico
# df = ...

# Caracterização geral
n_registros, n_variaveis = df.shape

# Identificação dos grupos associados ao hash
n_hash_distintos = df["hash"].nunique(dropna=True)

frequencia_hash = (
    df.loc[df["hash"].notna(), "hash"]
    .value_counts()
)

n_hash_repetidos = frequencia_hash.gt(1).sum()
n_registros_hash_repetidos = frequencia_hash[
    frequencia_hash.gt(1)
].sum()

# Conversão da data de cadastro
datas = pd.to_datetime(
    df["data_de_cadastro"].astype("string").str.strip(),
    errors="coerce",
    format="mixed",
    dayfirst=True
)

# Síntese estrutural
caracterizacao = pd.DataFrame({
    "indicador": [
        "Quantidade de registros",
        "Quantidade de variáveis",
        "Quantidade de hashes distintos",
        "Hashes associados a múltiplos registros",
        "Registros pertencentes a hashes repetidos",
        "Hashes ausentes",
        "Registros integralmente duplicados",
        "Data inicial dos registros",
        "Data final dos registros"
    ],
    "valor": [
        n_registros,
        n_variaveis,
        n_hash_distintos,
        n_hash_repetidos,
        int(n_registros_hash_repetidos),
        int(df["hash"].isna().sum()),
        int(df.duplicated().sum()),
        datas.min(),
        datas.max()
    ]
})

display(caracterizacao)


,indicador,valor
0,Quantidade de registros,8401
1,Quantidade de variáveis,62
2,Quantidade de hashes distintos,1263
3,Hashes associados a múltiplos registros,1166
4,Registros pertencentes a hashes repetidos,8304
5,Hashes ausentes,0
6,Registros integralmente duplicados,0
7,Data inicial dos registros,2021-01-02 23:06:00
8,Data final dos registros,2025-12-30 15:31:58


In [ ]:
pd.set_option("display.max_rows", None)

columns = df_raw.columns.difference(["hash", "data_de_cadastro"])

for col in columns:
    display(df_raw[col].value_counts() \
            .sort_index(ascending=True) \
            .to_frame())

,count
canal_de_atendimento,
CARTA,1
E-MAIL,641
PORTAL,29
PROCESSO SEI,84
TELEFÔNICO,6091
TELEGRAM,14
WEBCHAT,46
WHATSAPP,1495


,count
cenario_da_violacao,
AMBIENTE DE LAZER/ESPORTE/ENTRETENIMENTO,37
AMBIENTE VIRTUAL (NO ÂMBITO DA INTERNET),333
"BAR, RESTAURANTE, LANCHONETE",18
CASA DA VÍTIMA,1225
CASA DE FAMILIARES,434
CASA DE TERCEIRO,66
CASA DO SUSPEITO,1088
CASA NOTURNA,6
CASA ONDE RESIDE A VÍTIMA E O SUSPEITO,1776


,count
deficiencia_da_vitima,
AUDITIVA/SURDEZ,2
AUTISMO,12
FÍSICA/MOTORA,3
MENTAL/INTELECTUAL,32
N/D,817
NÃO,966
NÃO INFORMADO,32
NÃO TEM DEFICIÊNCIA,122
TEM DEFICIÊNCIA MAS NÃO SABE QUAL,2


,count
deficiencia_do_suspeito,
FÍSICA/MOTORA,3
MENTAL/INTELECTUAL,2
N/D,801
NÃO,971
NÃO INFORMADO,26
NÃO TEM DEFICIÊNCIA,143
VISUAL,1


,count
deficiencia_relacionada_a_doenca_rara,
NÃO,177
NÃO SABE INFORMAR,10


,count
deficiencia_relacionada_a_doenca_rara_suspeito,
NÃO,217
NÃO SABE INFORMAR,730


... (50 outputs left out)

,count
vinculo_orgao_pj_do_suspeito,


,count
violacao,
DIREITOS CIVIS E POLÍTICOS>ACESSO À INFORMAÇÃO,16
DIREITOS CIVIS E POLÍTICOS>CULTURAL,4
DIREITOS CIVIS E POLÍTICOS>LIVRE EXERCÍCIO DO PODER FAMILIAR,7
DIREITOS CIVIS E POLÍTICOS>MEMÓRIA E VERDADE,8
DIREITOS CIVIS E POLÍTICOS>NACIONALIDADE,2
DIREITOS CIVIS E POLÍTICOS>PROPRIEDADE>PATRIMÔNIO GENÉTICO,11
DIREITOS CIVIS E POLÍTICOS>PROPRIEDADE>PATRIMÔNIO IMATERIAL,3
DIREITOS CIVIS E POLÍTICOS>PROPRIEDADE>PATRIMÔNIO MATERIAL,6
DIREITOS CIVIS E POLÍTICOS>RETENÇÃO DE DOCUMENTOS,41


,count
vitima_especificacao_etnia,
N/D,933
NÃO INFORMADO,28
OUTROS,30


,count
vitima_preso_a,
N/D,740
NÃO,4655
NÃO INFORMADO,543


## 2.2 Define quality rules and canonical comparison functions

The comparison key normalizes Unicode, trims and collapses whitespace, converts text to uppercase, and removes accents. It is used **only to detect equivalent variants**. Original values remain unchanged.

The textual missing-value vocabulary should be reviewed against the source system's official codebook.


In [ ]:
REAL_MISSING = {
    "",
    "NULO",
    "NULL",
    "N/D",
    "N/A",
    "nan",
    "NaN",
    "NAN",
    None,
}

INFORMATIONAL_MISSING = {
    "NÃO INFORMADO",
    "NÃO SABE INFORMAR",
    "NÃO SABE",
    "DENUNCIANTE NÃO SOUBE INFORMAR",
}

OPERATIONAL_MISSING = {
    "ATENDIMENTO INTERROMPIDO",
    "ATENDIMENTO INTERRPOMPIDO",
}

NOT_APPLICABLE = {
    "NÃO SE APLICA",
    "NÃO SE APLICA - VÍTIMA COMUNIDADE/FAMÍLIA",
}

PARTIAL_INFORMATION = {
    "CRIANÇA/ADOLESCENTE IDADE NÃO INFORMADA",
    "IDOSO IDADE NÃO INFORMADA",
}

UNKOWN_VALUE = {
    "DESCONHECIDO(A)",
    "DESCONHECIDO (A)"
}


TEXT_MISSING_TOKENS = REAL_MISSING | INFORMATIONAL_MISSING | OPERATIONAL_MISSING | NOT_APPLICABLE | UNKOWN_VALUE

# TEXT_MISSING_TOKENS = {
#     "", "N/D", "N/A", "NA", "NAN", "NaN", "nan", "NULL", "NULO", "NONE", None,
#     "NÃO INFORMADO", "NAO SABE INFORMAR", "NAO SE APLICA", "NAO APLICAVEL",
#     "NAO SABE INFORMAR", "IGNORADO", "IGNORADA", "DESCONHECIDO", "DESCONHECIDA",
#     "INDEFINIDO", "INDEFINIDA", "NI", "NSA", 
#     "ATENDIMENTO "
# }

UF_CODES = {
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO", "MA", "MT",
    "MS", "MG", "PA", "PB", "PR", "PE", "PI", "RJ", "RN", "RS", "RO",
    "RR", "SC", "SP", "SE", "TO"
}

CONDITIONAL_COLUMNS = {
    "motivacao", "sl_vitima_naturalidade", "sl_vitima_naturalizado_uf",
    "sl_vitima_naturalizado_municipio", "deficiencia_da_vitima",
    "deficiencia_relacionada_a_doenca_rara", "vitima_preso_a",
    "profissao_da_vitima", "grau_de_instrucao_da_vitima", "religiao_da_vitima",
    "etnia_da_vitima", "faixa_de_renda_da_vitima", "sl_suspeito_naturalidade",
    "sl_suspeito_naturalizado_uf", "sl_suspeito_naturalizado_municipio",
    "deficiencia_do_suspeito", "deficiencia_relacionada_a_doenca_rara_suspeito",
    "suspeito_preso", "profissao_do_suspeito", "grau_de_instrucao_do_suspeito",
    "religiao_do_suspeito", "suspeito_etnia", "faixa_de_renda_do_suspeito",
    "vinculo_orgao_pj_do_suspeito", "sl_suspeito_ramo",
    "suspeito_especificacao_etnia", "vitima_especificacao_etnia"
}

CORE_COLUMNS = {
    "hash", "data_de_cadastro", "canal_de_atendimento", "denuncia_emergencial",
    "cenario_da_violacao", "pais", "grupo_vulneravel", "sl_quantidade_vitimas",
    "relacao_vitima_suspeito", "natureza_juridica_do_suspeito", "violacao"
}


def strip_accents(text):
    return "".join(ch for ch in unicodedata.normalize("NFKD", text) if not unicodedata.combining(ch))


def comparison_key(value):
    if pd.isna(value):
        return ""
    text = unicodedata.normalize("NFKC", str(value))
    text = re.sub(r"\s+", " ", text.strip()).upper()
    return strip_accents(text)


CANONICAL_MISSING_TOKENS = {comparison_key(x) for x in TEXT_MISSING_TOKENS}


In [ ]:
# for col in df_raw.columns.difference(["hash", "data_de_cadastro"]):
#     print(df_raw[col].replace(TEXT_MISSING_TOKENS, pd.NA).value_counts(dropna=False).sort_index())

## 2.3 Baseline schema and type assessment

This section flags unexpected types without automatically coercing the source. The expected high-level structure is one datetime field, one integer quantity field, and predominantly categorical text fields.


In [ ]:
EXPECTED_TYPE_RULES = {
    "data_de_cadastro": "datetime",
    "sl_quantidade_vitimas": "numeric",
}

if df is not None:
    type_rows = []
    for column in df.columns:
        observed = str(df[column].dtype)
        expected = EXPECTED_TYPE_RULES.get(column, "text_or_category")
        if expected == "datetime":
            parsed = pd.to_datetime(df[column], errors="coerce")
            conforming = int(parsed.notna().sum())
            invalid = int((df[column].notna() & parsed.isna()).sum())
        elif expected == "numeric":
            parsed = pd.to_numeric(df[column], errors="coerce")
            conforming = int(parsed.notna().sum())
            invalid = int((df[column].notna() & parsed.isna()).sum())
        else:
            conforming = int(df[column].notna().sum())
            invalid = 0
        type_rows.append({
            "column": column, "observed_dtype": observed, 
            "expected_role": expected,
            "non_null": int(df[column].notna().sum()), 
            "parseable_or_conforming": conforming,
            "invalid_for_expected_type": invalid
        })
    type_assessment = pd.DataFrame(type_rows)
    display(type_assessment)


,column,observed_dtype,expected_role,non_null,parseable_or_conforming,invalid_for_expected_type
0,hash,str,text_or_category,8401,8401,0
1,data_de_cadastro,str,datetime,8401,8401,0
2,canal_de_atendimento,str,text_or_category,8401,8401,0
3,denuncia_emergencial,str,text_or_category,8401,8401,0
4,denunciante,str,text_or_category,8401,8401,0
5,cenario_da_violacao,str,text_or_category,8401,8401,0
6,pais,str,text_or_category,8401,8401,0
7,uf,str,text_or_category,7828,7828,0
8,municipio,str,text_or_category,7810,7810,0
9,frequencia,str,text_or_category,8401,8401,0


## 2.4 Real and semantic missingness

**Real missingness** consists of `NaN`, `None`, or `pd.NA`. **Semantic missingness** consists of text tokens whose meaning is absence of information. Effective missingness combines both without modifying the source.

Interpretation must distinguish:

- **Core fields**, where high missingness is generally concerning
- **Conditional fields**, where non-applicability may be legitimate
- **Analytical fields**, where missingness reduces the scope of subgroup analysis


In [ ]:
def semantic_missing_masks(data):
    masks = {}
    details = []
    text_columns = data.select_dtypes(include=["object", "string", "category"]).columns
    for column in text_columns:
        keys = data[column].map(comparison_key)
        mask = data[column].notna() & keys.isin(CANONICAL_MISSING_TOKENS)
        masks[column] = mask
        if mask.any():
            for value, count in data.loc[mask, column].astype("string").value_counts(dropna=False).items():
                details.append({
                    "column": column, 
                    "raw_value": value, 
                    "count": int(count),
                    "percent_rows": round(count / len(data) * 100, 2)
                })
    return masks, pd.DataFrame(details)


if df is not None:
    text_missing_masks, semantic_missing_detail = semantic_missing_masks(df)
    completeness_rows = []
    for column in df.columns:
        real = int(df[column].isna().sum())
        semantic = int(text_missing_masks.get(column, pd.Series(False, index=df.index)).sum())
        effective = real + semantic
        if column in CORE_COLUMNS:
            field_class = "core"
        elif column in CONDITIONAL_COLUMNS:
            field_class = "conditional"
        else:
            field_class = "analytical"
        completeness_rows.append({
            "column": column, 
            "field_class": field_class,
            "real_missing": real, 
            "semantic_missing": semantic,
            "effective_missing": effective,
            "effective_missing_pct": round(effective / len(df) * 100, 2),
            "completeness_pct": round((1 - effective / len(df)) * 100, 2)
        })
    completeness_profile = pd.DataFrame(completeness_rows).sort_values("effective_missing_pct", ascending=False)
    display(completeness_profile)
    display(semantic_missing_detail)


,column,field_class,real_missing,semantic_missing,effective_missing,effective_missing_pct,completeness_pct
57,vinculo_orgao_pj_do_suspeito,conditional,8401,0,8401,100.00,0.00
59,suspeito_especificacao_etnia,conditional,7427,954,8381,99.76,0.24
60,vitima_especificacao_etnia,conditional,7410,961,8371,99.64,0.36
34,etnia_da_vitima,conditional,7422,915,8337,99.24,0.76
32,religiao_da_vitima,conditional,7186,1147,8333,99.19,0.81
55,suspeito_etnia,conditional,7416,915,8331,99.17,0.83
35,faixa_de_renda_da_vitima,conditional,7438,857,8295,98.74,1.26
56,faixa_de_renda_do_suspeito,conditional,7454,785,8239,98.07,1.93
25,deficiencia_relacionada_a_doenca_rara,conditional,8214,10,8224,97.89,2.11
53,religiao_do_suspeito,conditional,7231,989,8220,97.85,2.15


,column,raw_value,count,percent_rows
0,denuncia_emergencial,N/D,1,0.01
1,cenario_da_violacao,NÃO SABE INFORMAR,242,2.88
2,uf,DENUNCIANTE NÃO SOUBE INFORMAR,773,9.20
3,uf,N/D,136,1.62
4,uf,ATENDIMENTO INTERROMPIDO,18,0.21
5,municipio,DENUNCIANTE NÃO SOUBE INFORMAR,781,9.30
6,municipio,N/D,136,1.62
7,municipio,ATENDIMENTO INTERROMPIDO,18,0.21
8,frequencia,NÃO SABE INFORMAR,1553,18.49
9,frequencia,N/D,17,0.20


In [ ]:

import pandas as pd
import numpy as np
import unicodedata
import re


N = len(df_raw)


def normalizar_categoria(valor):
    if pd.isna(valor):
        return None

    texto = str(valor).strip().upper()

    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(
        c for c in texto
        if not unicodedata.combining(c)
    )

    return re.sub(r"\s+", " ", texto)


CODIGOS_DESCONHECIMENTO = {
    "N/D",
    "ND",
    "NAO INFORMADO",
    "NAO SABE INFORMAR",
    "NAO SABE",
    "DESCONHECIDO",
    "DESCONHECIDO(A)",
}

CODIGOS_ADMINISTRATIVOS = {
    "DENUNCIANTE NAO SOUBE INFORMAR",
    "ATENDIMENTO INTERROMPIDO",
    "ATENDIMENTO INTERRPOMPIDO",
}

CODIGOS_NAO_APLICAVEL = {
    "NAO SE APLICA",
    "NAO SE APLICA - VITIMA COMUNIDADE/FAMILIA",
}


def gerar_matriz_ausencias(data):
    resultados = []

    for coluna in data.columns:
        valores = data[coluna]
        normalizados = valores.map(normalizar_categoria)

        nan = valores.isna()
        desconhecimento = normalizados.isin(
            CODIGOS_DESCONHECIMENTO
        )
        administrativos = normalizados.isin(
            CODIGOS_ADMINISTRATIVOS
        )
        nao_aplicavel = normalizados.isin(
            CODIGOS_NAO_APLICAVEL
        )

        substantivos = ~(
            nan
            | desconhecimento
            | administrativos
            | nao_aplicavel
        )

        resultados.append({
            "variavel": coluna,
            "n_total": len(data),
            "n_nan": int(nan.sum()),
            "pct_nan": round(nan.mean() * 100, 2),
            "n_desconhecido": int(desconhecimento.sum()),
            "pct_desconhecido": round(
                desconhecimento.mean() * 100, 2
            ),
            "n_administrativo": int(administrativos.sum()),
            "pct_administrativo": round(
                administrativos.mean() * 100, 2
            ),
            "n_nao_aplicavel": int(nao_aplicavel.sum()),
            "pct_nao_aplicavel": round(
                nao_aplicavel.mean() * 100, 2
            ),
            "n_informacao_substantiva": int(substantivos.sum()),
            "pct_informacao_substantiva": round(
                substantivos.mean() * 100, 2
            ),
        })

    return (
        pd.DataFrame(resultados)
        .sort_values("pct_nan", ascending=False)
        .reset_index(drop=True)
    )


matriz_ausencias = gerar_matriz_ausencias(df_raw)

display(matriz_ausencias)

# Exportar para documentar a auditoria.
matriz_ausencias.to_excel(
    "matriz_ausencias_disque100.xlsx",
    index=False
)


,variavel,n_total,n_nan,pct_nan,n_desconhecido,pct_desconhecido,n_administrativo,pct_administrativo,n_nao_aplicavel,pct_nao_aplicavel,n_informacao_substantiva,pct_informacao_substantiva
0,vinculo_orgao_pj_do_suspeito,8401,8401,100.00,0,0.00,0,0.00,0,0.00,0,0.00
1,deficiencia_relacionada_a_doenca_rara,8401,8214,97.77,10,0.12,0,0.00,0,0.00,177,2.11
2,sl_vitima_naturalizado_municipio,8401,8189,97.48,0,0.00,15,0.18,0,0.00,197,2.34
3,sl_vitima_naturalizado_uf,8401,8141,96.91,0,0.00,52,0.62,0,0.00,208,2.48
4,sl_suspeito_naturalizado_municipio,8401,8131,96.79,0,0.00,31,0.37,0,0.00,239,2.84
5,sl_vitima_naturalidade,8401,8083,96.21,0,0.00,0,0.00,0,0.00,318,3.79
6,sl_suspeito_naturalizado_uf,8401,8046,95.77,0,0.00,98,1.17,0,0.00,257,3.06
7,sl_suspeito_naturalidade,8401,8015,95.41,0,0.00,0,0.00,0,0.00,386,4.59
8,faixa_de_renda_do_suspeito,8401,7454,88.73,785,9.34,0,0.00,0,0.00,162,1.93
9,grau_de_instrucao_do_suspeito,8401,7454,88.73,740,8.81,0,0.00,0,0.00,207,2.46


In [ ]:

import pandas as pd

# DataFrame analítico
# df = ...

# Caracterização geral
n_registros, n_variaveis = df.shape

# Identificação dos grupos associados ao hash
n_hash_distintos = df["hash"].nunique(dropna=True)

frequencia_hash = (
    df.loc[df["hash"].notna(), "hash"]
    .value_counts()
)

n_hash_repetidos = frequencia_hash.gt(1).sum()
n_registros_hash_repetidos = frequencia_hash[
    frequencia_hash.gt(1)
].sum()

# Conversão da data de cadastro
datas = pd.to_datetime(
    df["data_de_cadastro"].astype("string").str.strip(),
    errors="coerce",
    format="mixed",
    dayfirst=True
)

# Síntese estrutural
caracterizacao = pd.DataFrame({
    "indicador": [
        "Quantidade de registros",
        "Quantidade de variáveis",
        "Quantidade de hashes distintos",
        "Hashes associados a múltiplos registros",
        "Registros pertencentes a hashes repetidos",
        "Hashes ausentes",
        "Registros integralmente duplicados",
        "Data inicial dos registros",
        "Data final dos registros"
    ],
    "valor": [
        n_registros,
        n_variaveis,
        n_hash_distintos,
        n_hash_repetidos,
        int(n_registros_hash_repetidos),
        int(df["hash"].isna().sum()),
        int(df.duplicated().sum()),
        datas.min(),
        datas.max()
    ]
})

display(caracterizacao)


,indicador,valor
0,Quantidade de registros,8401
1,Quantidade de variáveis,62
2,Quantidade de hashes distintos,1263
3,Hashes associados a múltiplos registros,1166
4,Registros pertencentes a hashes repetidos,8304
5,Hashes ausentes,0
6,Registros integralmente duplicados,0
7,Data inicial dos registros,2021-01-02 23:06:00
8,Data final dos registros,2025-12-30 15:31:58


In [ ]:
preditores_ausencia = [
    "canal_de_atendimento",
    "denunciante",
    "sl_vitima_cadastro",
    "natureza_juridica_do_suspeito",
    "denuncia_emergencial",
    "grupo_vulneravel",
]

In [ ]:

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    StratifiedKFold,
    StratifiedGroupKFold,
    cross_val_score
)
from sklearn.metrics import roc_auc_score
from sklearn.base import clone


# ============================================================
# 1. CONFIGURAÇÕES
# ============================================================

# DataFrame original
# df = ...

# Variáveis cuja ausência será avaliada
alvos = [
    "faixa_de_renda_da_vitima",
    "violacao",
    "religiao_da_vitima",
    "profissao_da_vitima",
    "faixa_etaria_do_suspeito",
    "municipio",
    "profissao_do_suspeito",
    "grau_de_instrucao_da_vitima",
    "municipio_da_vitima",
    "sl_suspeito_municipio",
    "orientacao_sexual_da_vitima",
    "raca_cor_do_suspeito",
    "motivacao",
]

COLUNA_GRUPO = "hash"

N_SPLITS = 5
RANDOM_STATE = 42

# Limite mínimo de frequência para manter categorias
MIN_FREQUENCIA = 10

# Variáveis que não devem ser usadas como preditoras
# Ajuste conforme a finalidade e a estrutura da base.
EXCLUIR_PREDITORES = [
    "hash"
]


# ============================================================
# 2. AUDITORIA INICIAL
# ============================================================

if COLUNA_GRUPO not in df.columns:
    raise ValueError(f"A coluna {COLUNA_GRUPO!r} não existe.")

alvos_inexistentes = [c for c in alvos if c not in df.columns]

if alvos_inexistentes:
    raise ValueError(f"Alvos inexistentes: {alvos_inexistentes}")

print("=" * 70)
print("AUDITORIA INICIAL")
print("=" * 70)

print(f"Dimensão original: {df.shape}")
print(f"Quantidade de hashes distintos: {df[COLUNA_GRUPO].nunique(dropna=True)}")
print(f"Hashes ausentes: {df[COLUNA_GRUPO].isna().sum()}")


# ============================================================
# 3. EXCLUIR VARIÁVEIS INTEGRALMENTE AUSENTES
# ============================================================

colunas_totalmente_ausentes = df.columns[df.isna().all()].tolist()

print("\nColunas integralmente ausentes:")
print(colunas_totalmente_ausentes)

df_padrao = df.drop(columns=colunas_totalmente_ausentes).copy()

print(f"\nDimensão após exclusão: {df_padrao.shape}")


# ============================================================
# 4. DISTRIBUIÇÃO DE AUSÊNCIAS POR REGISTRO
# ============================================================

quantidade_ausentes_por_registro = df_padrao.isna().sum(axis=1)

distribuicao_ausencias = (
    quantidade_ausentes_por_registro
    .value_counts()
    .sort_index()
    .rename_axis("quantidade_variaveis_ausentes")
    .reset_index(name="quantidade_registros")
)

distribuicao_ausencias["percentual_registros"] = (
    distribuicao_ausencias["quantidade_registros"]
    / len(df_padrao) * 100
)

print("\n" + "=" * 70)
print("DISTRIBUIÇÃO DE AUSÊNCIAS POR REGISTRO")
print("=" * 70)

display(distribuicao_ausencias)

print(
    "\nRegistros sem qualquer ausência:",
    (quantidade_ausentes_por_registro == 0).sum()
)

print(
    "Registros com pelo menos uma ausência:",
    (quantidade_ausentes_por_registro > 0).sum()
)


# ============================================================
# 5. MATRIZ DE COAUSÊNCIAS ENTRE VARIÁVEIS
# ============================================================

indicadores_nan = df_padrao.isna().astype("int8")

matriz_coausencias = indicadores_nan.T.dot(indicadores_nan)

print("\n" + "=" * 70)
print("MATRIZ DE COAUSÊNCIAS")
print("=" * 70)

display(matriz_coausencias)


# ============================================================
# 6. MATRIZ DE PROPORÇÃO DE COAUSÊNCIAS
# ============================================================

# Proporção em relação ao total de registros
matriz_proporcao_coausencias = matriz_coausencias / len(df_padrao)

display(matriz_proporcao_coausencias)


# ============================================================
# 7. PREPARAR PREDITORES
# ============================================================

# Não utilizar:
# - a própria variável-alvo;
# - hash como preditor;
# - variáveis totalmente ausentes.

preditores_base = [
    c for c in df_padrao.columns
    if c not in set(alvos + EXCLUIR_PREDITORES)
]

# Converter datas em atributos temporais, quando aplicável.
# Evita utilizar timestamp bruto como variável categórica.
if "data_de_cadastro" in preditores_base:
    data = pd.to_datetime(
        df_padrao["data_de_cadastro"],
        errors="coerce",
        dayfirst=True
    )

    df_padrao["ano_cadastro"] = data.dt.year.astype("float")
    df_padrao["mes_cadastro"] = data.dt.month.astype("float")

    preditores_base.remove("data_de_cadastro")

    for coluna in ["ano_cadastro", "mes_cadastro"]:
        if coluna not in preditores_base:
            preditores_base.append(coluna)


# ============================================================
# 8. FUNÇÃO DE PREPARAÇÃO DO MODELO
# ============================================================

def criar_pipeline(X):

    numericas = X.select_dtypes(
        include=["number", "bool"]
    ).columns.tolist()

    categoricas = [
        c for c in X.columns if c not in numericas
    ]

    transformadores = []

    if numericas:
        pipeline_numerico = Pipeline([
            ("imputacao", SimpleImputer(strategy="median")),
            ("escala", StandardScaler())
        ])

        transformadores.append(
            ("num", pipeline_numerico, numericas)
        )

    if categoricas:
        pipeline_categorico = Pipeline([
            (
                "imputacao",
                SimpleImputer(
                    strategy="constant",
                    fill_value="__AUSENCIA_NO_PREDITOR__"
                )
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    min_frequency=MIN_FREQUENCIA
                )
            )
        ])

        transformadores.append(
            ("cat", pipeline_categorico, categoricas)
        )

    preprocessor = ColumnTransformer(
        transformers=transformadores,
        remainder="drop"
    )

    modelo = LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        solver="liblinear",
        random_state=RANDOM_STATE
    )

    return Pipeline([
        ("preprocessamento", preprocessor),
        ("modelo", modelo)
    ])


# ============================================================
# 9. FUNÇÃO DE AVALIAÇÃO DA AUSÊNCIA
# ============================================================

def avaliar_previsibilidade_ausencia_grupo(
    data,
    alvo,
    preditores,
    n_splits=5
):

    # O alvo precisa ser excluído dos preditores.
    colunas_X = [
        c for c in preditores
        if c != alvo and c in data.columns
    ]

    # Indicador de ausência técnica:
    # 1 = ausente; 0 = preenchido.
    y = data[alvo].isna().astype(int)

    # Verificar variabilidade do alvo.
    if y.nunique() < 2:
        return {
            "variavel": alvo,
            "resultado": "Sem variabilidade",
            "n_ausentes": int(y.sum()),
            "pct_ausencia": y.mean() * 100,
            "auc_media": np.nan,
            "auc_dp": np.nan,
            "n_grupos": data[COLUNA_GRUPO].nunique()
        }

    # Excluir linhas sem identificador de grupo.
    mascara = data[COLUNA_GRUPO].notna()

    X = data.loc[mascara, colunas_X].copy()
    y = y.loc[mascara]
    grupos = data.loc[mascara, COLUNA_GRUPO]

    # Eliminar preditores sem variabilidade.
    colunas_validas = [
        c for c in X.columns
        if X[c].nunique(dropna=True) > 1
    ]

    X = X[colunas_validas]

    if X.shape[1] == 0:
        return {
            "variavel": alvo,
            "resultado": "Sem preditores válidos",
            "n_ausentes": int(y.sum()),
            "pct_ausencia": y.mean() * 100,
            "auc_media": np.nan,
            "auc_dp": np.nan,
            "n_grupos": grupos.nunique()
        }

    # Número de grupos com cada classe.
    grupos_por_classe = {
        classe: grupos[y == classe].nunique()
        for classe in y.unique()
    }

    menor_grupos = min(grupos_por_classe.values())

    if menor_grupos < n_splits:
        return {
            "variavel": alvo,
            "resultado": (
                "Grupos insuficientes para "
                f"{n_splits} folds"
            ),
            "n_ausentes": int(y.sum()),
            "pct_ausencia": y.mean() * 100,
            "auc_media": np.nan,
            "auc_dp": np.nan,
            "n_grupos": grupos.nunique()
        }

    cv = StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    pipeline = criar_pipeline(X)

    aucs = []

    for treino, teste in cv.split(X, y, groups=grupos):

        X_treino = X.iloc[treino]
        X_teste = X.iloc[teste]

        y_treino = y.iloc[treino]
        y_teste = y.iloc[teste]

        # Evitar folds sem as duas classes.
        if y_treino.nunique() < 2 or y_teste.nunique() < 2:
            continue

        modelo = clone(pipeline)

        modelo.fit(X_treino, y_treino)

        probabilidades = modelo.predict_proba(X_teste)[:, 1]

        auc = roc_auc_score(y_teste, probabilidades)

        aucs.append(auc)

    if not aucs:
        resultado = "Não foi possível calcular AUC"
        auc_media = np.nan
        auc_dp = np.nan
    else:
        resultado = "Avaliado"
        auc_media = np.mean(aucs)
        auc_dp = np.std(aucs, ddof=1) if len(aucs) > 1 else 0

    return {
        "variavel": alvo,
        "resultado": resultado,
        "n_ausentes": int(y.sum()),
        "pct_ausencia": y.mean() * 100,
        "auc_media": auc_media,
        "auc_dp": auc_dp,
        "n_grupos": grupos.nunique(),
        "n_folds_validos": len(aucs)
    }


# ============================================================
# 10. EXECUTAR VALIDAÇÃO AGRUPADA POR HASH
# ============================================================

resultados_grupo = []

for alvo in alvos:

    resultado = avaliar_previsibilidade_ausencia_grupo(
        data=df_padrao,
        alvo=alvo,
        preditores=preditores_base,
        n_splits=N_SPLITS
    )

    resultados_grupo.append(resultado)

diagnostico_grupo = pd.DataFrame(resultados_grupo)

diagnostico_grupo = diagnostico_grupo.sort_values(
    "auc_media",
    ascending=False,
    na_position="last"
).reset_index(drop=True)

print("\n" + "=" * 70)
print("PREVISIBILIDADE DA AUSÊNCIA - VALIDAÇÃO AGRUPADA POR HASH")
print("=" * 70)

display(diagnostico_grupo)


# ============================================================
# 11. COMPARAÇÃO COM VALIDAÇÃO ALEATÓRIA
# ============================================================

def avaliar_previsibilidade_ausencia_aleatoria(
    data,
    alvo,
    preditores,
    n_splits=5
):

    colunas_X = [
        c for c in preditores
        if c != alvo and c in data.columns
    ]

    y = data[alvo].isna().astype(int)

    if y.nunique() < 2:
        return np.nan, np.nan

    X = data[colunas_X].copy()

    colunas_validas = [
        c for c in X.columns
        if X[c].nunique(dropna=True) > 1
    ]

    X = X[colunas_validas]

    if X.shape[1] == 0:
        return np.nan, np.nan

    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    pipeline = criar_pipeline(X)

    aucs = cross_val_score(
        pipeline,
        X,
        y,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1
    )

    return aucs.mean(), aucs.std(ddof=1)


resultados_aleatorios = []

for alvo in alvos:

    auc_media, auc_dp = avaliar_previsibilidade_ausencia_aleatoria(
        data=df_padrao,
        alvo=alvo,
        preditores=preditores_base,
        n_splits=N_SPLITS
    )

    resultados_aleatorios.append({
        "variavel": alvo,
        "auc_aleatoria": auc_media,
        "auc_dp_aleatoria": auc_dp
    })

diagnostico_aleatorio = pd.DataFrame(resultados_aleatorios)


# ============================================================
# 12. CONSOLIDAR RESULTADOS
# ============================================================

comparacao = diagnostico_grupo.merge(
    diagnostico_aleatorio,
    on="variavel",
    how="left"
)

comparacao["diferenca_auc"] = (
    comparacao["auc_aleatoria"] -
    comparacao["auc_media"]
)

comparacao = comparacao.sort_values(
    "auc_media",
    ascending=False,
    na_position="last"
)

print("\n" + "=" * 70)
print("COMPARAÇÃO: VALIDAÇÃO ALEATÓRIA X AGRUPADA")
print("=" * 70)

display(comparacao)


# ============================================================
# 13. EXPORTAR RESULTADOS
# ============================================================

distribuicao_ausencias.to_csv(
    f"{OUTPUT_DIR}/distribuicao_ausencias_por_registro.csv",
    index=False,
    encoding="utf-8-sig"
)

matriz_coausencias.to_csv(
    f"{OUTPUT_DIR}/matriz_coausencias.csv",
    encoding="utf-8-sig"
)

matriz_proporcao_coausencias.to_csv(
    f"{OUTPUT_DIR}/matriz_proporcao_coausencias.csv",
    encoding="utf-8-sig"
)

diagnostico_grupo.to_csv(
    f"{OUTPUT_DIR}/diagnostico_ausencia_validacao_hash.csv",
    index=False,
    encoding="utf-8-sig"
)

diagnostico_aleatorio.to_csv(
    f"{OUTPUT_DIR}/diagnostico_ausencia_validacao_aleatoria.csv",
    index=False,
    encoding="utf-8-sig"
)

comparacao.to_csv(
    f"{OUTPUT_DIR}/comparacao_validacao_ausencia.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nArquivos CSV exportados com sucesso.")


AUDITORIA INICIAL
Dimensão original: (8401, 62)
Quantidade de hashes distintos: 1263
Hashes ausentes: 0

Colunas integralmente ausentes:
['vinculo_orgao_pj_do_suspeito']

Dimensão após exclusão: (8401, 61)

DISTRIBUIÇÃO DE AUSÊNCIAS POR REGISTRO


,quantidade_variaveis_ausentes,quantidade_registros,percentual_registros
0,0,43,0.511844
1,1,106,1.261755
2,2,23,0.273777
3,3,32,0.380907
4,4,57,0.678491
5,5,101,1.202238
6,6,161,1.916439
7,7,165,1.964052
8,8,197,2.344959
9,9,62,0.738007



Registros sem qualquer ausência: 43
Registros com pelo menos uma ausência: 8358

MATRIZ DE COAUSÊNCIAS


,hash,data_de_cadastro,canal_de_atendimento,denuncia_emergencial,denunciante,cenario_da_violacao,pais,uf,municipio,frequencia,inicio_das_violacoes,sl_quantidade_vitimas,grupo_vulneravel,motivacao,relacao_vitima_suspeito,sl_vitima_cadastro,genero_da_vitima,orientacao_sexual_da_vitima,faixa_etaria_da_vitima,nacionalidade_da_vitima,sl_vitima_naturalidade,sl_vitima_naturalizado_uf,sl_vitima_naturalizado_municipio,deficiencia_da_vitima,doenca_rara_da_vitima,deficiencia_relacionada_a_doenca_rara,vitima_preso_a,pais_da_vitima,uf_da_vitima,municipio_da_vitima,profissao_da_vitima,grau_de_instrucao_da_vitima,religiao_da_vitima,raca_cor_da_vitima,etnia_da_vitima,faixa_de_renda_da_vitima,natureza_juridica_do_suspeito,genero_do_suspeito,orientacao_sexual_do_suspeito,faixa_etaria_do_suspeito,suspeito_nacionalidade,sl_suspeito_naturalidade,sl_suspeito_naturalizado_uf,sl_suspeito_naturalizado_municipio,deficiencia_do_suspeito,doenca_rara_do_suspeito,deficiencia_relacionada_a_doenca_rara_suspeito,suspeito_preso,pais_do_suspeito,uf_do_suspeito,sl_suspeito_municipio,profissao_do_suspeito,grau_de_instrucao_do_suspeito,religiao_do_suspeito,raca_cor_do_suspeito,suspeito_etnia,faixa_de_renda_do_suspeito,sl_suspeito_ramo,suspeito_especificacao_etnia,vitima_especificacao_etnia,violacao
hash,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
data_de_cadastro,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
canal_de_atendimento,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
denuncia_emergencial,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
denunciante,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
cenario_da_violacao,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
pais,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
uf,0,0,0,0,0,0,0,61,61,0,0,0,0,-30,0,0,0,52,0,0,61,61,61,0,4,61,52,0,-8,3,41,57,61,0,61,61,0,36,36,36,0,61,61,61,-4,36,61,33,0,23,23,50,61,61,-103,61,61,25,61,61,0
municipio,0,0,0,0,0,0,0,61,79,0,0,0,0,-30,0,0,0,52,0,0,79,79,79,12,4,79,52,0,10,21,59,75,79,0,79,79,0,36,36,36,0,79,79,79,8,36,79,33,0,41,41,50,79,79,-103,79,79,43,79,79,0
frequencia,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


,hash,data_de_cadastro,canal_de_atendimento,denuncia_emergencial,denunciante,cenario_da_violacao,pais,uf,municipio,frequencia,inicio_das_violacoes,sl_quantidade_vitimas,grupo_vulneravel,motivacao,relacao_vitima_suspeito,sl_vitima_cadastro,genero_da_vitima,orientacao_sexual_da_vitima,faixa_etaria_da_vitima,nacionalidade_da_vitima,sl_vitima_naturalidade,sl_vitima_naturalizado_uf,sl_vitima_naturalizado_municipio,deficiencia_da_vitima,doenca_rara_da_vitima,deficiencia_relacionada_a_doenca_rara,vitima_preso_a,pais_da_vitima,uf_da_vitima,municipio_da_vitima,profissao_da_vitima,grau_de_instrucao_da_vitima,religiao_da_vitima,raca_cor_da_vitima,etnia_da_vitima,faixa_de_renda_da_vitima,natureza_juridica_do_suspeito,genero_do_suspeito,orientacao_sexual_do_suspeito,faixa_etaria_do_suspeito,suspeito_nacionalidade,sl_suspeito_naturalidade,sl_suspeito_naturalizado_uf,sl_suspeito_naturalizado_municipio,deficiencia_do_suspeito,doenca_rara_do_suspeito,deficiencia_relacionada_a_doenca_rara_suspeito,suspeito_preso,pais_do_suspeito,uf_do_suspeito,sl_suspeito_municipio,profissao_do_suspeito,grau_de_instrucao_do_suspeito,religiao_do_suspeito,raca_cor_do_suspeito,suspeito_etnia,faixa_de_renda_do_suspeito,sl_suspeito_ramo,suspeito_especificacao_etnia,vitima_especificacao_etnia,violacao
hash,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
data_de_cadastro,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
canal_de_atendimento,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
denuncia_emergencial,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
denunciante,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
cenario_da_violacao,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.00


PREVISIBILIDADE DA AUSÊNCIA - VALIDAÇÃO AGRUPADA POR HASH


,variavel,resultado,n_ausentes,pct_ausencia,auc_media,auc_dp,n_grupos,n_folds_validos
0,municipio,Avaliado,591,7.034877,1.000000,0.000000,1263,5
1,municipio_da_vitima,Avaliado,641,7.630044,0.999430,0.001075,1263,5
2,sl_suspeito_municipio,Avaliado,505,6.011189,0.998352,0.002415,1263,5
3,faixa_de_renda_da_vitima,Avaliado,7438,88.537079,0.997276,0.003073,1263,5
4,faixa_etaria_do_suspeito,Avaliado,2879,34.269730,0.995091,0.003820,1263,5
5,religiao_da_vitima,Avaliado,7186,85.537436,0.992665,0.010990,1263,5
6,profissao_do_suspeito,Avaliado,2798,33.305559,0.992285,0.003466,1263,5
7,orientacao_sexual_da_vitima,Avaliado,2371,28.222831,0.991781,0.006463,1263,5
8,profissao_da_vitima,Avaliado,7026,83.632901,0.980323,0.018118,1263,5
9,grau_de_instrucao_da_vitima,Avaliado,6743,80.264254,0.975402,0.005347,1263,5



COMPARAÇÃO: VALIDAÇÃO ALEATÓRIA X AGRUPADA


,variavel,resultado,n_ausentes,pct_ausencia,auc_media,auc_dp,n_grupos,n_folds_validos,auc_aleatoria,auc_dp_aleatoria,diferenca_auc
0,municipio,Avaliado,591,7.034877,1.000000,0.000000,1263,5,1.000000,0.000000,0.000000
1,municipio_da_vitima,Avaliado,641,7.630044,0.999430,0.001075,1263,5,1.000000,0.000000,0.000570
2,sl_suspeito_municipio,Avaliado,505,6.011189,0.998352,0.002415,1263,5,0.999946,0.000037,0.001594
3,faixa_de_renda_da_vitima,Avaliado,7438,88.537079,0.997276,0.003073,1263,5,0.999654,0.000767,0.002378
4,faixa_etaria_do_suspeito,Avaliado,2879,34.269730,0.995091,0.003820,1263,5,0.999607,0.000471,0.004516
5,religiao_da_vitima,Avaliado,7186,85.537436,0.992665,0.010990,1263,5,0.999848,0.000272,0.007183
6,profissao_do_suspeito,Avaliado,2798,33.305559,0.992285,0.003466,1263,5,0.998912,0.000577,0.006627
7,orientacao_sexual_da_vitima,Avaliado,2371,28.222831,0.991781,0.006463,1263,5,0.999159,0.000773,0.007378
8,profissao_da_vitima,Avaliado,7026,83.632901,0.980323,0.018118,1263,5,0.999544,0.000544,0.019221
9,grau_de_instrucao_da_vitima,Avaliado,6743,80.264254,0.975402,0.005347,1263,5,0.997687,0.003362,0.022286



Arquivos CSV exportados com sucesso.


### Interactive missingness chart

The 40% and 80% lines are review aids, not automatic deletion rules. Conditional fields should be evaluated with their activation rules and applicable denominator.


In [ ]:
completeness_profile.sort_values("effective_missing_pct", ascending=True)

,column,field_class,real_missing,semantic_missing,effective_missing,effective_missing_pct,completeness_pct
36,natureza_juridica_do_suspeito,core,0,0,0,0.00,100.00
48,pais_do_suspeito,analytical,0,0,0,0.00,100.00
6,pais,core,0,0,0,0.00,100.00
11,sl_quantidade_vitimas,core,0,0,0,0.00,100.00
15,sl_vitima_cadastro,analytical,0,0,0,0.00,100.00
27,pais_da_vitima,analytical,0,0,0,0.00,100.00
12,grupo_vulneravel,core,0,0,0,0.00,100.00
2,canal_de_atendimento,core,0,0,0,0.00,100.00
1,data_de_cadastro,core,0,0,0,0.00,100.00
4,denunciante,analytical,0,0,0,0.00,100.00


In [ ]:
if df is not None:
    plot_data = completeness_profile.sort_values("effective_missing_pct", ascending=True)
    fig = px.bar(
        plot_data, x="effective_missing_pct", y="column", orientation="h",
        color="field_class", height=max(750, len(plot_data) * 20),
        hover_data=["real_missing", "semantic_missing", "completeness_pct"],
        title="Percentual de ausência efetiva de dados por coluna",
        labels={"effective_missing_pct": "Ausência Efetiva (%)", "column": "Coluna"},
        
    )
    fig.add_vline(x=50, line_dash="dash", line_color="#f59e0b", annotation_text="50% revisão")
    fig.add_vline(x=80, line_dash="dash", line_color="#dc2626", annotation_text="80% severo")
    fig.update_layout(template="plotly_white")
    fig.show()


## 2.5 Missingness patterns across rows

A row-level burden score reveals records with unusually little information. This is not a deletion rule because some sections may be structurally inapplicable.


In [ ]:
if df is not None:
    effective_missing_matrix = df.isna().copy()
    for column, mask in text_missing_masks.items():
        effective_missing_matrix[column] = effective_missing_matrix[column] | mask

    row_quality = pd.DataFrame(index=df.index)
    row_quality["missing_fields"] = effective_missing_matrix.sum(axis=1)
    row_quality["missing_pct"] = (row_quality["missing_fields"] / df.shape[1] * 100).round(2)
    display(row_quality.describe(percentiles=[.5, .75, .9, .95, .99]))

    fig = px.histogram(
        row_quality, x="missing_pct", nbins=40,
        title="Distribution of effective missingness across records",
        labels={"missing_pct": "Missing fields per record (%)", "count": "Records"}
    )
    fig.update_layout(template="plotly_white")
    fig.show()


,missing_fields,missing_pct
count,8401.000000,8401.000000
mean,29.868349,48.175151
std,4.516246,7.283915
min,10.000000,16.130000
50%,30.000000,48.390000
75%,33.000000,53.230000
90%,35.000000,56.450000
95%,37.000000,59.680000
99%,41.000000,66.130000
max,46.000000,74.190000


## 2.6 Whitespace and invisible-character inconsistencies

The checks detect leading or trailing whitespace, repeated internal spaces, and tabs or line breaks. Examples are retained to support an auditable correction map.


In [ ]:
if df is not None:
    whitespace_rows = []
    for column in df.select_dtypes(include=["object", "string", "category"]).columns:
        s = df[column].astype("string")
        checks = {
            "leading_or_trailing_whitespace": (s.str.match(r"^\s") | s.str.match(r".*\s$")).fillna(False),
            "repeated_internal_whitespace": s.str.contains(r"\s{2,}", regex=True, na=False),
            "tab_or_line_break": s.str.contains(r"[\t\r\n]", regex=True, na=False),
            "non_breaking_space": s.str.contains("\u00a0", regex=False, na=False),
        }
        for issue, mask in checks.items():
            if mask.any():
                whitespace_rows.append({
                    "column": column, "issue": issue, "records": int(mask.sum()),
                    "percent_rows": round(mask.mean() * 100, 2),
                    "examples": " | ".join(s[mask].dropna().unique()[:8])
                })
    whitespace_issues = pd.DataFrame(whitespace_rows)
    display(whitespace_issues)


,column,issue,records,percent_rows,examples
0,faixa_de_renda_da_vitima,leading_or_trailing_whitespace,7,0.08,3 A 5 SM
1,violacao,leading_or_trailing_whitespace,70,0.83,LIBERDADE>DIREITOS INDIVIDUAIS>CONDIÇÃO ANÁLOGA À DE ESCRAVO>TRANSPORTAR TRABALHADOR PARA FINS DE EXPLORAÇÃO


## 2.7 Case, accent, Unicode, and spacing variants

Values that share the same canonical key but have different original forms are candidates for standardization. Examples include `Não informado`, `NAO INFORMADO`, and `NÃO  INFORMADO`.


In [ ]:
if df is not None:
    variant_rows = []
    for column in df.select_dtypes(include=["object", "string", "category"]).columns:
        groups = defaultdict(set)
        for value in df[column].dropna().astype(str).unique():
            groups[comparison_key(value)].add(value)
        for canonical, variants in groups.items():
            if canonical and len(variants) > 1:
                counts = df[column].astype("string").value_counts()
                total = sum(int(counts.get(v, 0)) for v in variants)
                variant_rows.append({
                    "column": column, "canonical_key": canonical,
                    "variant_count": len(variants), "records": total,
                    "variants": " | ".join(sorted(variants)[:12])
                })
    category_variants = pd.DataFrame(variant_rows).sort_values(
        ["records", "variant_count"], ascending=False
    ) if variant_rows else pd.DataFrame(columns=["column", "canonical_key", "variant_count", "records", "variants"])
    display(category_variants)

,column,canonical_key,variant_count,records,variants
0,violacao,LIBERDADE>DIREITOS INDIVIDUAIS>CONDICAO ANALOGA A DE ESCRAVO>TRANSPORTAR TRABALHADOR PARA FINS DE EXPLORACAO,2,80,LIBERDADE>DIREITOS INDIVIDUAIS>CONDIÇÃO ANÁLOGA À DE ESCRAVO>TRANSPORTAR TRABALHADOR PARA FINS DE EXPLORAÇÃO | LIBERDADE>DIREITOS INDIVI...


### Interactive inconsistency chart


In [ ]:
if df is not None and not category_variants.empty:
    variant_summary = category_variants.groupby("column", as_index=False).agg(
        inconsistent_groups=("canonical_key", "size"),
        affected_records=("records", "sum")
    ).sort_values("affected_records")
    fig = px.bar(
        variant_summary, x="affected_records", y="column", orientation="h",
        color="inconsistent_groups", height=max(500, len(variant_summary) * 28),
        title="Potential categorical inconsistencies by column"
    )
    fig.update_layout(template="plotly_white")
    fig.show()


## 2.8 Duplicate structures and dataset grain

Three distinct problems are evaluated:

1. Exact duplicate rows
2. Repeated `hash` values
3. Repeated `hash` values with conflicting attributes

Repeated identifiers are not automatically errors. They may reveal the true one-to-many structure of the source.


In [ ]:
if df is not None:
    exact_duplicate_mask = df.duplicated(keep=False)
    duplicate_summary = [{
        "scope": "exact rows", "affected_rows": int(exact_duplicate_mask.sum()),
        "duplicate_groups": int(df[exact_duplicate_mask].drop_duplicates().shape[0]) if exact_duplicate_mask.any() else 0
    }]

    hash_conflicts = pd.DataFrame()
    if "hash" in df.columns:
        repeated_hash_mask = df["hash"].notna() & df["hash"].duplicated(keep=False)
        duplicate_summary.append({
            "scope": "repeated hash", "affected_rows": int(repeated_hash_mask.sum()),
            "duplicate_groups": int(df.loc[repeated_hash_mask, "hash"].nunique())
        })
        conflict_columns = [
            c for c in ["data_de_cadastro", "violacao", "grupo_vulneravel",
                        "sl_quantidade_vitimas", "relacao_vitima_suspeito",
                        "genero_da_vitima", "faixa_etaria_da_vitima",
                        "genero_do_suspeito", "faixa_etaria_do_suspeito"] if c in df.columns
        ]
        repeated = df.loc[repeated_hash_mask, ["hash"] + conflict_columns]
        if not repeated.empty:
            hash_conflicts = repeated.groupby("hash")[conflict_columns].nunique(dropna=False)
            hash_conflicts["conflicting_columns"] = (hash_conflicts > 1).sum(axis=1)
            hash_conflicts = hash_conflicts.reset_index().sort_values("conflicting_columns", ascending=False)

    duplicate_report = pd.DataFrame(duplicate_summary)
    display(duplicate_report)
    display(hash_conflicts.head(20))


,scope,affected_rows,duplicate_groups
0,exact rows,0,0
1,repeated hash,8304,1166


,hash,data_de_cadastro,violacao,grupo_vulneravel,sl_quantidade_vitimas,relacao_vitima_suspeito,genero_da_vitima,faixa_etaria_da_vitima,genero_do_suspeito,faixa_etaria_do_suspeito,conflicting_columns
815,B451B8BCF178136079FD3A66E170BEB581695A56D03DD111E347E46E989ADC00,1,18,2,1,1,1,1,2,2,4
149,1F4F3DEC641985020954D79317E38110F2AEB73FFDB97429A7E540063CB28734,1,9,1,1,1,2,2,1,1,3
115,19B15BC5207028920B41AB948CAC898D72FFD21509449D7FD7FBD894AD6B3CC8,1,9,1,1,1,2,2,1,1,3
104,16190F3A785AF8B94BF799C414534E18A618D8039C03C3FD5097AC2343322180,1,3,2,1,1,1,2,1,1,3
218,2F80468ED9106B37FC5429B1EE590F03737233A6D4FEBBE24CAB8990076169BC,1,2,1,1,1,1,1,2,2,3
472,684593F1D07A57846297148CF00DC30231A65449C45FD4A15C6136563AA68528,1,9,1,1,1,2,2,1,1,3
430,5E6C98B8F9655A163EE4C08F5D96A74DE674ED3262601033D7E9E76F531258F4,1,5,1,1,1,1,1,2,2,3
237,33E1D1CD49E8C9A577C063FD6035FA1BCF8D023DCBC6C36F9330FEC0D440E4DA,1,9,1,1,1,2,2,1,1,3
878,C3C7E9F26F11783E7463E3A8CA66BB1A494A105F0FB06643073725A18F48AF82,1,10,2,1,1,1,1,1,1,2
877,C3B96F47B6FF3CAE74A71817469CA898F2857E0DDC90621F324424A5665F4557,1,17,2,1,1,1,1,1,1,2


## 2.9 Domain constraints

Built-in rules validate Brazilian UF codes. Other domains must be populated from the official data dictionary rather than inferred from observed values. Add approved categories to `ALLOWED_DOMAINS` when they are available.


In [ ]:
ALLOWED_DOMAINS = {
    "uf": UF_CODES,
    "uf_da_vitima": UF_CODES,
    "uf_do_suspeito": UF_CODES,
    "sl_vitima_naturalizado_uf": UF_CODES,
    "sl_suspeito_naturalizado_uf": UF_CODES,
    # Examples to complete from the official codebook:
    # "denuncia_emergencial": {"SIM", "NAO"},
    # "genero_da_vitima": {...},
    # "genero_do_suspeito": {...},
}

if df is not None:
    domain_rows = []
    for column, allowed in ALLOWED_DOMAINS.items():
        if column not in df.columns:
            continue
        canonical_allowed = {comparison_key(x) for x in allowed}
        keys = df[column].map(comparison_key)
        missing_mask = df[column].isna() | text_missing_masks.get(column, pd.Series(False, index=df.index))
        invalid_mask = ~missing_mask & ~keys.isin(canonical_allowed)
        for value, count in df.loc[invalid_mask, column].astype("string").value_counts().items():
            domain_rows.append({
                "column": column, "invalid_value": value, "records": int(count),
                "percent_rows": round(count / len(df) * 100, 2)
            })
    domain_violations = pd.DataFrame(domain_rows)
    display(domain_violations)


""


## 2.10 Geographic consistency

The analysis checks whether Brazilian records have valid UF values and whether municipality fields are populated when UF is available. Municipality-name validity requires an authoritative IBGE municipality reference table and is therefore reported as a future enrichment rule rather than guessed.


In [ ]:
if df is not None:
    geo_rows = []
    pairs = [
        ("pais", "uf", "municipio", "report"),
        ("pais_da_vitima", "uf_da_vitima", "municipio_da_vitima", "victim"),
        ("pais_do_suspeito", "uf_do_suspeito", "sl_suspeito_municipio", "suspect"),
    ]
    for country_col, uf_col, municipality_col, entity in pairs:
        if country_col not in df.columns:
            continue
        country = df[country_col].map(comparison_key)
        brazil = country.isin({"BRASIL", "BRAZIL", "BR"})
        if uf_col in df.columns:
            uf_missing = df[uf_col].isna() | text_missing_masks.get(uf_col, pd.Series(False, index=df.index))
            uf_invalid = ~uf_missing & ~df[uf_col].map(comparison_key).isin(UF_CODES)
            geo_rows.extend([
                {"entity": entity, "rule": "Brazil requires UF", "violations": int((brazil & uf_missing).sum())},
                {"entity": entity, "rule": "UF must be valid", "violations": int(uf_invalid.sum())},
            ])
        if municipality_col in df.columns and uf_col in df.columns:
            municipality_missing = df[municipality_col].isna() | text_missing_masks.get(municipality_col, pd.Series(False, index=df.index))
            valid_uf = df[uf_col].map(comparison_key).isin(UF_CODES)
            geo_rows.append({
                "entity": entity, "rule": "Valid UF requires municipality review",
                "violations": int((valid_uf & municipality_missing).sum())
            })
    geographic_checks = pd.DataFrame(geo_rows)
    geographic_checks["percent_rows"] = (geographic_checks["violations"] / len(df) * 100).round(2)
    display(geographic_checks)


,entity,rule,violations,percent_rows
0,report,Brazil requires UF,105,1.25
1,report,UF must be valid,0,0.00
2,report,Valid UF requires municipality review,26,0.31
3,victim,Brazil requires UF,93,1.11
4,victim,UF must be valid,0,0.00
5,victim,Valid UF requires municipality review,116,1.38
6,suspect,Brazil requires UF,75,0.89
7,suspect,UF must be valid,0,0.00
8,suspect,Valid UF requires municipality review,111,1.32


## 2.11 Date and quantity validity

The notebook checks parsing, future registration dates, positive victim counts, and integer-valued victim counts. Extreme positive quantities are flagged with the IQR rule for manual review, not automatically removed.


In [ ]:
if df is not None:
    validity_rows = []
    date_diagnostics = pd.DataFrame()
    if "data_de_cadastro" in df.columns:
        parsed_date = pd.to_datetime(df["data_de_cadastro"], errors="coerce")
        unparseable = df["data_de_cadastro"].notna() & parsed_date.isna()
        future = parsed_date.notna() & (parsed_date > pd.Timestamp.now())
        validity_rows.extend([
            {"rule": "registration date must be parseable", "violations": int(unparseable.sum())},
            {"rule": "registration date must not be future", "violations": int(future.sum())},
        ])
        date_diagnostics = pd.DataFrame({
            "minimum": [parsed_date.min()], "maximum": [parsed_date.max()],
            "unparseable": [int(unparseable.sum())], "future": [int(future.sum())]
        })

    quantity_diagnostics = pd.DataFrame()
    if "sl_quantidade_vitimas" in df.columns:
        quantity = pd.to_numeric(df["sl_quantidade_vitimas"], errors="coerce")
        invalid_numeric = df["sl_quantidade_vitimas"].notna() & quantity.isna()
        non_positive = quantity.notna() & (quantity <= 0)
        non_integer = quantity.notna() & ((quantity % 1) != 0)
        q1, q3 = quantity.quantile([.25, .75])
        iqr = q3 - q1
        upper_fence = q3 + 1.5 * iqr
        high_outlier = quantity.notna() & (quantity > upper_fence)
        validity_rows.extend([
            {"rule": "victim quantity must be numeric", "violations": int(invalid_numeric.sum())},
            {"rule": "victim quantity must be positive", "violations": int(non_positive.sum())},
            {"rule": "victim quantity must be integer", "violations": int(non_integer.sum())},
            {"rule": "victim quantity above IQR review fence", "violations": int(high_outlier.sum())},
        ])
        quantity_diagnostics = pd.DataFrame({
            "minimum": [quantity.min()], "q1": [q1], "median": [quantity.median()],
            "q3": [q3], "maximum": [quantity.max()], "iqr_upper_fence": [upper_fence]
        })

    validity_checks = pd.DataFrame(validity_rows)
    validity_checks["percent_rows"] = (validity_checks["violations"] / len(df) * 100).round(2)
    display(validity_checks)
    display(date_diagnostics)
    display(quantity_diagnostics)


,rule,violations,percent_rows
0,registration date must be parseable,0,0.00
1,registration date must not be future,0,0.00
2,victim quantity must be numeric,0,0.00
3,victim quantity must be positive,0,0.00
4,victim quantity must be integer,0,0.00
5,victim quantity above IQR review fence,963,11.46


,minimum,maximum,unparseable,future
0,2021-01-05 08:28:00,2025-12-30 15:31:58,0,0


,minimum,q1,median,q3,maximum,iqr_upper_fence
0,1,1.0,1.0,2.0,60,3.5


## 2.12 Cross-field business rules

These rules identify logically suspicious combinations. They require domain review before correction because hotline forms may permit exceptional cases.


In [ ]:
def missing_mask(data, column, semantic_masks):
    return data[column].isna() | semantic_masks.get(column, pd.Series(False, index=data.index))

if df is not None:
    business_rows = []
    def add_rule(rule, mask, explanation):
        mask = mask.fillna(False)
        business_rows.append({
            "rule": rule, "violations": int(mask.sum()),
            "percent_rows": round(mask.mean() * 100, 2), "explanation": explanation
        })

    if {"doenca_rara_da_vitima", "deficiencia_relacionada_a_doenca_rara"}.issubset(df.columns):
        rare_no = df["doenca_rara_da_vitima"].map(comparison_key).isin({"NAO", "N"})
        related_present = ~missing_mask(df, "deficiencia_relacionada_a_doenca_rara", text_missing_masks)
        add_rule("rare disease = no but related disability is populated", rare_no & related_present,
                 "Review whether the related-disability field should be applicable.")

    if {"doenca_rara_do_suspeito", "deficiencia_relacionada_a_doenca_rara_suspeito"}.issubset(df.columns):
        rare_no = df["doenca_rara_do_suspeito"].map(comparison_key).isin({"NAO", "N"})
        related_present = ~missing_mask(df, "deficiencia_relacionada_a_doenca_rara_suspeito", text_missing_masks)
        add_rule("suspect rare disease = no but related disability is populated", rare_no & related_present,
                 "Review conditional applicability.")

    if {"nacionalidade_da_vitima", "sl_vitima_naturalizado_uf"}.issubset(df.columns):
        brazilian = df["nacionalidade_da_vitima"].map(comparison_key).isin({"BRASILEIRA", "BRASILEIRO", "BRASIL"})
        naturalized_uf_present = ~missing_mask(df, "sl_vitima_naturalizado_uf", text_missing_masks)
        add_rule("Brazilian victim with naturalization UF populated", brazilian & naturalized_uf_present,
                 "May be valid for naturalized Brazilians; verify field semantics.")

    if {"suspeito_nacionalidade", "sl_suspeito_naturalizado_uf"}.issubset(df.columns):
        brazilian = df["suspeito_nacionalidade"].map(comparison_key).isin({"BRASILEIRA", "BRASILEIRO", "BRASIL"})
        naturalized_uf_present = ~missing_mask(df, "sl_suspeito_naturalizado_uf", text_missing_masks)
        add_rule("Brazilian suspect with naturalization UF populated", brazilian & naturalized_uf_present,
                 "May be valid for naturalized Brazilians; verify field semantics.")

    if {"etnia_da_vitima", "vitima_especificacao_etnia"}.issubset(df.columns):
        ethnicity_missing = missing_mask(df, "etnia_da_vitima", text_missing_masks)
        specification_present = ~missing_mask(df, "vitima_especificacao_etnia", text_missing_masks)
        add_rule("victim ethnicity specification without ethnicity", ethnicity_missing & specification_present,
                 "Specification normally requires a parent ethnicity value.")

    if {"suspeito_etnia", "suspeito_especificacao_etnia"}.issubset(df.columns):
        ethnicity_missing = missing_mask(df, "suspeito_etnia", text_missing_masks)
        specification_present = ~missing_mask(df, "suspeito_especificacao_etnia", text_missing_masks)
        add_rule("suspect ethnicity specification without ethnicity", ethnicity_missing & specification_present,
                 "Specification normally requires a parent ethnicity value.")

    business_rule_checks = pd.DataFrame(business_rows)
    display(business_rule_checks)


,rule,violations,percent_rows,explanation
0,rare disease = no but related disability is populated,0,0.00,Review whether the related-disability field should be applicable.
1,suspect rare disease = no but related disability is populated,0,0.00,Review conditional applicability.
2,Brazilian victim with naturalization UF populated,208,2.48,May be valid for naturalized Brazilians; verify field semantics.
3,Brazilian suspect with naturalization UF populated,257,3.06,May be valid for naturalized Brazilians; verify field semantics.
4,victim ethnicity specification without ethnicity,16,0.19,Specification normally requires a parent ethnicity value.
5,suspect ethnicity specification without ethnicity,6,0.07,Specification normally requires a parent ethnicity value.


## 2.13 Interactive quality dashboard

This summary combines the principal diagnostic classes. Counts are indicators for prioritization and must not be summed as unique defective records because a record may violate multiple rules.


In [ ]:
if df is not None:
    dashboard_rows = []
    dashboard_rows.append({"quality_dimension": "Completeness", "indicator": "Effective missing cells", "count": int(completeness_profile["effective_missing"].sum())})
    dashboard_rows.append({"quality_dimension": "Uniqueness", "indicator": "Exact duplicate rows", "count": int(exact_duplicate_mask.sum())})
    if "hash" in df.columns:
        dashboard_rows.append({"quality_dimension": "Uniqueness", "indicator": "Rows with repeated hash", "count": int((df["hash"].notna() & df["hash"].duplicated(keep=False)).sum())})
    dashboard_rows.append({"quality_dimension": "Consistency", "indicator": "Whitespace issue instances", "count": int(whitespace_issues["records"].sum()) if not whitespace_issues.empty else 0})
    dashboard_rows.append({"quality_dimension": "Consistency", "indicator": "Variant-group affected records", "count": int(category_variants["records"].sum()) if not category_variants.empty else 0})
    dashboard_rows.append({"quality_dimension": "Validity", "indicator": "Domain violation instances", "count": int(domain_violations["records"].sum()) if not domain_violations.empty else 0})
    dashboard_rows.append({"quality_dimension": "Validity", "indicator": "Date/quantity rule violations", "count": int(validity_checks["violations"].sum()) if not validity_checks.empty else 0})
    dashboard_rows.append({"quality_dimension": "Consistency", "indicator": "Cross-field rule violations", "count": int(business_rule_checks["violations"].sum()) if not business_rule_checks.empty else 0})
    quality_dashboard = pd.DataFrame(dashboard_rows)

    fig = px.bar(
        quality_dashboard, x="count", y="indicator", orientation="h",
        color="quality_dimension", text="count", height=520,
        title="Data-quality indicators by dimension"
    )
    fig.update_layout(template="plotly_white", yaxis={"categoryorder": "total ascending"})
    fig.show()
    display(quality_dashboard)


,quality_dimension,indicator,count
0,Completeness,Effective missing cells,250924
1,Uniqueness,Exact duplicate rows,0
2,Uniqueness,Rows with repeated hash,8304
3,Consistency,Whitespace issue instances,77
4,Consistency,Variant-group affected records,80
5,Validity,Domain violation instances,0
6,Validity,Date/quantity rule violations,963
7,Consistency,Cross-field rule violations,487


## 2.14 Quality issue register and prioritization

Priority is based on field role and affected records:

- **Critical:** identifier, date, violation, or core report fields
- **High:** geographic validity, duplicate structure, or broad category inconsistency
- **Medium:** analytical subgroup fields
- **Review:** conditional fields requiring an applicability denominator

The register supports decisions but does not automatically delete columns or rows.


In [ ]:
if df is not None:
    issue_rows = []
    for _, row in completeness_profile.iterrows():
        if row["effective_missing"] == 0:
            continue
        if row["field_class"] == "core":
            priority = "critical" if row["effective_missing_pct"] > 0 else "low"
        elif row["field_class"] == "conditional":
            priority = "review"
        else:
            priority = "high" if row["effective_missing_pct"] >= 40 else "medium"
        issue_rows.append({
            "dimension": "completeness", "column_or_scope": row["column"],
            "issue": "effective missingness", "affected": int(row["effective_missing"]),
            "percent_rows": row["effective_missing_pct"], "priority": priority,
            "recommended_action": "Assess applicability and analytical impact; do not delete automatically."
        })

    if not whitespace_issues.empty:
        for _, row in whitespace_issues.iterrows():
            issue_rows.append({
                "dimension": "consistency", "column_or_scope": row["column"],
                "issue": row["issue"], "affected": int(row["records"]),
                "percent_rows": row["percent_rows"], "priority": "medium",
                "recommended_action": "Normalize Unicode and whitespace while retaining an audit map."
            })

    if not domain_violations.empty:
        for column, group in domain_violations.groupby("column"):
            issue_rows.append({
                "dimension": "validity", "column_or_scope": column,
                "issue": "values outside approved domain", "affected": int(group["records"].sum()),
                "percent_rows": round(group["records"].sum()/len(df)*100, 2), "priority": "high",
                "recommended_action": "Recode only with an authoritative mapping; otherwise quarantine for review."
            })

    quality_issue_register = pd.DataFrame(issue_rows)
    priority_order = pd.CategoricalDtype(["critical", "high", "medium", "review", "low"], ordered=True)
    if not quality_issue_register.empty:
        quality_issue_register["priority"] = quality_issue_register["priority"].astype(priority_order)
        quality_issue_register = quality_issue_register.sort_values(["priority", "affected"], ascending=[True, False])
    display(quality_issue_register)


,dimension,column_or_scope,issue,affected,percent_rows,priority,recommended_action
36,completeness,relacao_vitima_suspeito,effective missingness,2103,25.03,critical,Assess applicability and analytical impact; do not delete automatically.
46,completeness,cenario_da_violacao,effective missingness,242,2.88,critical,Assess applicability and analytical impact; do not delete automatically.
48,completeness,violacao,effective missingness,190,2.26,critical,Assess applicability and analytical impact; do not delete automatically.
50,completeness,denuncia_emergencial,effective missingness,1,0.01,critical,Assess applicability and analytical impact; do not delete automatically.
25,completeness,raca_cor_do_suspeito,effective missingness,5100,60.71,high,Assess applicability and analytical impact; do not delete automatically.
27,completeness,faixa_etaria_do_suspeito,effective missingness,4167,49.60,high,Assess applicability and analytical impact; do not delete automatically.
28,completeness,raca_cor_da_vitima,effective missingness,4019,47.84,high,Assess applicability and analytical impact; do not delete automatically.
30,completeness,orientacao_sexual_da_vitima,effective missingness,3707,44.13,high,Assess applicability and analytical impact; do not delete automatically.
31,completeness,orientacao_sexual_do_suspeito,effective missingness,2678,31.88,medium,Assess applicability and analytical impact; do not delete automatically.
32,completeness,inicio_das_violacoes,effective missingness,2574,30.64,medium,Assess applicability and analytical impact; do not delete automatically.


## 2.15 Optional normalization proposal

This creates a **proposed** normalized copy rather than changing `df_raw`. It:

- applies Unicode NFKC normalization
- trims leading and trailing whitespace
- collapses repeated whitespace
- standardizes text to uppercase
- converts approved textual missing tokens to `pd.NA`

Accent removal is **not** applied to display values because accents carry correct Portuguese spelling. It is used only for comparison and matching.


In [ ]:
def normalize_text_value(value):
    if pd.isna(value):
        return pd.NA
    text = unicodedata.normalize("NFKC", str(value))
    text = re.sub(r"\s+", " ", text.strip()).upper()
    if comparison_key(text) in CANONICAL_MISSING_TOKENS:
        return pd.NA
    return text

if df is not None:
    df_proposed = df.copy(deep=True)
    text_columns = df_proposed.select_dtypes(include=["object", "string", "category"]).columns
    for column in text_columns:
        df_proposed[column] = df_proposed[column].map(normalize_text_value).astype("string")

    normalization_summary = pd.DataFrame({
        "column": text_columns,
        "changed_cells": [int((df[column].astype("string").fillna("<NA>") != df_proposed[column].astype("string").fillna("<NA>")).sum()) for column in text_columns]
    }).sort_values("changed_cells", ascending=False)
    display(normalization_summary)


,column,changed_cells
32,raca_cor_da_vitima,3809
50,profissao_do_suspeito,2801
10,inicio_das_violacoes,2574
46,suspeito_preso,2141
13,relacao_vitima_suspeito,2103
9,frequencia,1570
28,municipio_da_vitima,1479
53,raca_cor_do_suspeito,1440
27,uf_da_vitima,1416
16,orientacao_sexual_da_vitima,1336


## 2.16 Before-versus-proposed comparison

This comparison quantifies the effect of standardization without asserting that every proposed change is substantively correct.


In [ ]:
if df is not None:
    comparison_rows = []
    for column in df.columns:
        before_unique = int(df[column].nunique(dropna=True))
        after_unique = int(df_proposed[column].nunique(dropna=True)) if column in df_proposed else before_unique
        before_missing = int(df[column].isna().sum())
        after_missing = int(df_proposed[column].isna().sum())
        comparison_rows.append({
            "column": column, "unique_before": before_unique, "unique_after": after_unique,
            "unique_reduction": before_unique - after_unique,
            "missing_before": before_missing, "missing_after": after_missing,
            "semantic_missing_converted": after_missing - before_missing
        })
    before_after = pd.DataFrame(comparison_rows).sort_values(
        ["semantic_missing_converted", "unique_reduction"], ascending=False
    )
    display(before_after)

    chart_data = before_after[before_after["unique_reduction"] > 0].sort_values("unique_reduction")
    if not chart_data.empty:
        fig = px.bar(
            chart_data, x="unique_reduction", y="column", orientation="h",
            title="Reduction in distinct categories after proposed normalization",
            labels={"unique_reduction": "Reduction in distinct values"}
        )
        fig.update_layout(template="plotly_white", height=max(450, len(chart_data)*28))
        fig.show()


,column,unique_before,unique_after,unique_reduction,missing_before,missing_after,semantic_missing_converted
33,raca_cor_da_vitima,7,5,2,210,4019,3809
51,profissao_do_suspeito,51,49,2,2798,5599,2801
10,inicio_das_violacoes,9,6,3,0,2574,2574
47,suspeito_preso,9,7,2,2888,5029,2141
14,relacao_vitima_suspeito,43,39,4,0,2103,2103
9,frequencia,7,5,2,0,1570,1570
29,municipio_da_vitima,301,297,4,641,2120,1479
54,raca_cor_do_suspeito,6,4,2,3660,5100,1440
28,uf_da_vitima,30,27,3,588,2004,1416
17,orientacao_sexual_da_vitima,10,8,2,2371,3707,1336


## 2.17 Export diagnostics

The normalized dataset is not exported automatically. Set `EXPORT_PROPOSED_DATA = True` only after reviewing the mappings and business rules.


In [ ]:
EXPORT_PROPOSED_DATA = False

if df is not None:
    exports = {
        "type_assessment.csv": type_assessment,
        "completeness_profile.csv": completeness_profile,
        "semantic_missing_detail.csv": semantic_missing_detail,
        "row_quality.csv": row_quality.reset_index(names="row_index"),
        "whitespace_issues.csv": whitespace_issues,
        "category_variants.csv": category_variants,
        "duplicate_report.csv": duplicate_report,
        "hash_conflicts.csv": hash_conflicts,
        "domain_violations.csv": domain_violations,
        "geographic_checks.csv": geographic_checks,
        "validity_checks.csv": validity_checks,
        "business_rule_checks.csv": business_rule_checks,
        "quality_dashboard.csv": quality_dashboard,
        "quality_issue_register.csv": quality_issue_register,
        "normalization_summary.csv": normalization_summary,
        "before_after_normalization.csv": before_after,
    }
    for filename, table in exports.items():
        table.to_csv(OUTPUT_DIR / filename, index=False, encoding="utf-8-sig")

    if EXPORT_PROPOSED_DATA:
        df_proposed.to_csv(OUTPUT_DIR / "proposed_normalized_data.csv", index=False, encoding="utf-8-sig")

    print(f"Diagnostics saved to: {OUTPUT_DIR.resolve()}")
    print("Proposed normalized data exported:", EXPORT_PROPOSED_DATA)


Diagnostics saved to: C:\Projetos\eda-disque100-completo\eda\outputs\step_02_data_quality
Proposed normalized data exported: False


## 2.18 Interpretation and decision checklist

Before moving to descriptive EDA, document the following decisions:

1. **Dataset grain:** Are repeated `hash` values structural or erroneous?
2. **Report denominator:** Will report counts use unique `hash` rather than rows?
3. **Semantic missingness:** Are all text tokens approved by the official codebook?
4. **Conditional denominators:** For each conditional field, what population is eligible to have a value?
5. **Category mappings:** Which accent, capitalization, and spacing variants can be merged safely?
6. **Geography:** Will municipality names be validated against an IBGE reference table?
7. **Domain constraints:** Which official allowed-value lists are available for gender, race/color, age group, channel, frequency, and violation?
8. **Outliers:** Which large `sl_quantidade_vitimas` values are plausible collective reports?
9. **Privacy:** Which exports require suppression, aggregation, or access control?
10. **Correction audit:** Will every correction record the column, original value, normalized value, rule, timestamp, and affected row count?

## Next step

**Step 3: Data Cleaning and Validation** should apply only approved mappings, preserve an immutable raw layer, produce a correction log, and compare distributions before and after cleaning. Substantive victim, suspect, violation, temporal, and geographic analyses should use the validated analytical dataset.
